In [1]:
import os
import numpy as np
import pandas as pd
import pathlib as plb
import matplotlib.pyplot as plt
import PIL
import tensorflow as tf
import cv2
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from tensorflow import keras
from keras import Model, Sequential
from keras.layers import Input, Flatten, Dropout,Dense, Conv2D, Convolution2D, Conv2DTranspose, MaxPooling2D, BatchNormalization, Activation, Concatenate, concatenate
from cv2 import imread, imwrite, imshow, resize
from datasets import load_dataset
from pandas import DataFrame
from PIL import Image
from numpy import arange, array, random, zeros, ones, stack, argmax

c:\Users\Lenovo\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
images_directory_path = r"C:\Users\Lenovo\Desktop\Courses\Programming\Programming docs\Datasets\YOLO-object-bounding-dataset\data\training_images"
label_file_path = r"C:\Users\Lenovo\Desktop\Courses\Programming\Programming docs\Datasets\YOLO-object-bounding-dataset\data\train_solution_bounding_boxes (1).csv"
image_size = 512
cell_size = 32
image_seprators = 16
cells_count = 256
image_width = 676
image_height = 380
height_scaler_const, weight_scaler_const = image_size/image_height, image_size/image_width
arc_height_scaler_const, arc_weight_scaler_const = 1/height_scaler_const, 1/weight_scaler_const
epochs = 24
batch_size = 16
dataset_len = 10

In [4]:
def get_grid(x):
    top, bottom, left, right = x['top'], x['bottom'], x['left'], x['right']
    
    delta_x =  right - left
    delta_y = bottom - top
    x_center = left + (delta_x/2)
    y_center = top + (delta_y/2)
    
    top_grid, bottom_grid, left_grid, right_grid = top/cell_size, bottom/cell_size, left/cell_size, right/cell_size

    x_center_grid, y_center_grid = x_center/cell_size, y_center/cell_size
    x_center_grid_index, y_center_grid_index = int(x_center_grid), int(y_center_grid)
    
    grid = np.zeros((16, 16, 5))
    grid[y_center_grid_index, x_center_grid_index, 0] = 1.0
    grid[y_center_grid_index, x_center_grid_index, 1] = top_grid
    grid[y_center_grid_index, x_center_grid_index, 2] = bottom_grid
    grid[y_center_grid_index, x_center_grid_index, 3] = left_grid
    grid[y_center_grid_index, x_center_grid_index, 4] = right_grid
    
    return grid

In [5]:
df = pd.read_csv(label_file_path)
df = DataFrame(df[['image', 'ymin', 'ymax', 'xmin', 'xmax']].to_numpy(), columns=['image', 'top', 'bottom', 'left', 'right'])
df.image = [[resize(imread(f"{images_directory_path}\\{fname}"), (image_size, image_size))] for fname in df.image]

In [6]:
sample_df = df[:10]
sample_df['image'] = list(stack(sample_df['image']).reshape((len(sample_df), image_size, image_size, 3)) / 255)
sample_df.top *= height_scaler_const
sample_df.bottom *= height_scaler_const
sample_df.left *= weight_scaler_const
sample_df.right *= weight_scaler_const
dataset_len = len(sample_df)

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_9460\2684462876.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sample_df['image'] = list(stack(sample_df['image']).reshape((len(sample_df), image_size, image_size, 3)) / 255)
C:\Users\Lenovo\AppData\Local\Temp\ipykernel_9460\2684462876.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sample_df.top *= height_scaler_const
C:\Users\Lenovo\AppData\Local\Temp\ipykernel_9460\2684462876.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a sl

In [7]:
sample_df['grid'] = sample_df[['top', 'bottom', 'left', 'right']].apply(lambda x : get_grid(x), axis=1)
x_train = stack(sample_df['image'])
y_train = stack(sample_df['grid'])

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_9460\3409950171.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  sample_df['grid'] = sample_df[['top', 'bottom', 'left', 'right']].apply(lambda x : get_grid(x), axis=1)


In [27]:
def get_weights(data):
    total_cells_count = dataset_len*image_seprators*image_seprators
    filled_cells_bool = (data[..., 0] == 1)
    filled_cells_count = np.sum(filled_cells_bool)
    empty_cells_count = total_cells_count - filled_cells_count
    
    filled_weight = total_cells_count / (2.0 * filled_cells_count)
    empty_weight = total_cells_count / (2.0 * empty_cells_count)
    
    return np.where(filled_cells_bool, filled_weight, empty_weight).astype(np.float32)

In [28]:
y_train_weights = get_weights(y_train)

In [10]:

input = Input(shape=(image_size, image_size, 3))
x = Conv2D(16, 3, activation='relu', padding='same')(input)
x = MaxPooling2D()(x)
x = Conv2D(32, 3, activation='relu', padding='same')(x)
x = MaxPooling2D()(x)
x = Conv2D(64, 3, activation='relu', padding='same')(x)
x = MaxPooling2D()(x)
x = Conv2D(128, 3, activation='relu', padding='same')(x)
x = MaxPooling2D()(x)
x = Conv2D(256, 3, activation='relu', padding='same')(x)
x = MaxPooling2D()(x)
x_position = Conv2D(4, 3, activation='linear', padding='same')(x)
x_objectness = Conv2D(1, 3, activation='sigmoid', padding='same')(x)
model = Model(inputs=input, outputs=[x_objectness, x_position])
output_names = list(model.output_names)

In [ ]:
def weighted_bce(y_true, y_pred):   
    bce = tf.keras.losses.binary_crossentropy(y_true, y_pred)
    return tf.reduce_mean( * y_train_weights)

def masked_mse(y_true, y_pred): 
    mask = y_true[..., 0:1]
    return tf.reduce_sum(mask * tf.square(y_true[..., 1:5] - y_pred)) / (tf.reduce_sum(mask) + 1e-6)

def masked_mae(y_true, y_pred):
    mask = y_true[..., 0:1]
    return tf.reduce_sum(mask * tf.abs(y_true[..., 1:5] - y_pred)) / (tf.reduce_sum(mask) + 1e-6)

def pixel_mae(y_train, y_pred): 
    return masked_mae(y_train, y_pred) * cell_size

def objectness_accuracy(y_true, y_pred, threshold=0.5): 
    pred_binary = tf.cast(y_pred > threshold, dtype=tf.float32)
    train_binary = tf.cast(y_true >= 1.0, dtype=tf.float32)
    accuracy_map = tf.cast((pred_binary == 1.0) & (train_binary == 1.0), dtype=tf.float32)
    total_objects_count = tf.reduce_sum(train_binary)
    return tf.reduce_sum(accuracy_map) / (total_objects_count + 1e-6)

In [32]:
model.compile(
    optimizer='adam',
    loss={output_names[0] : weighted_bce, output_names[1] : masked_mse}, 
    metrics={output_names[0] : [objectness_accuracy], output_names[1] : [masked_mse, masked_mae, pixel_mae]},
    )

In [33]:
history = model.fit(
    x_train,
    {output_names[0]: y_train[..., 0:1], output_names[1]: y_train},
    batch_size=16,
    epochs=6,
)

Epoch 1/6
1/1 ━━━━━━━━━━━━━━━━━━━━ 8s 8s/step - conv2d_5_loss: 38.8011 - conv2d_5_masked_mae: 10.8797 - conv2d_5_masked_mse: 38.8011 - conv2d_5_pixel_mae: 348.1520 - conv2d_6_loss: 0.2741 - conv2d_6_objectness_accuracy: 0.0000e+00 - loss: 39.0753
Epoch 2/6
1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step - conv2d_5_loss: 49.3500 - conv2d_5_masked_mae: 12.0117 - conv2d_5_masked_mse: 49.3500 - conv2d_5_pixel_mae: 384.3755 - conv2d_6_loss: 0.7276 - conv2d_6_objectness_accuracy: 0.0000e+00 - loss: 50.0776
Epoch 3/6
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step - conv2d_5_loss: 17.9133 - conv2d_5_masked_mae: 6.3373 - conv2d_5_masked_mse: 17.9133 - conv2d_5_pixel_mae: 202.7920 - conv2d_6_loss: 0.3245 - conv2d_6_objectness_accuracy: 0.0000e+00 - loss: 18.2377
Epoch 4/6
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step - conv2d_5_loss: 24.4024 - conv2d_5_masked_mae: 7.9542 - conv2d_5_masked_mse: 24.4024 - conv2d_5_pixel_mae: 254.5346 - conv2d_6_loss: 0.3347 - conv2d_6_objectness_accuracy: 0.0000e+00 - loss: 24.7371
Epoch 5/6
1/1 

np.float32(1.0)